# LIF File Analysis Pipeline

이 노트북은 `./data` 폴더의 `.lif` 파일을 자동으로 찾고, channel stack 저장, MIP 시각화, histogram/Fourier QC, Z-stack GIF 생성을 수행합니다.
아래 셀들은 기능별로 분리되어 있으므로 필요한 단계만 다시 실행할 수 있습니다.


## 0. Imports

공통 라이브러리와 notebook 표시 도구를 불러옵니다.


In [ ]:
import os
import json
import base64
from pathlib import Path
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import HTML, display

from readlif.reader import LifFile
from skimage import exposure, io
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


## 1. 컬러맵 정의

채널별 표시 색상과 macOS 리소스 파일 정리 유틸리티를 정의합니다.


In [ ]:
cmap_blue = LinearSegmentedColormap.from_list("dapi", ["black", "blue"])
cmap_green = LinearSegmentedColormap.from_list("reference", ["black", "green"])
cmap_red = LinearSegmentedColormap.from_list("target", ["black", "red"])


def cleanup_dot_underscore_files(target_dir, verbose=False):
    target_dir = Path(target_dir)
    removed_files = []

    if not target_dir.exists():
        return removed_files

    for file_path in target_dir.rglob("._*"):
        if not file_path.is_file():
            continue
        try:
            file_path.unlink()
            removed_files.append(file_path)
        except Exception as e:
            if verbose:
                print(f"[WARN] failed to remove {file_path}: {e}")

    if verbose and removed_files:
        print(f"[INFO] removed {len(removed_files)} ._ files from {target_dir}")

    return removed_files


## 2. metadata 파싱

LIF 파일 내부 series/ROI metadata를 읽고 여러 파일을 순회하는 함수를 정의합니다.


In [ ]:
def inspect_single_lif(lif_path, verbose=True):
    lif_path = Path(lif_path)

    if not lif_path.exists():
        print(f"[ERROR] File not found: {lif_path}")
        return None

    try:
        lif = LifFile(str(lif_path))

        if verbose:
            print(f"\nTarget File: {lif_path.name}")
            print(f"Full Path   : {lif_path}")
            print(f"Total number of images (ROI/Series): {lif.num_images}")
            print("=" * 70)

        file_results = []

        for i in range(lif.num_images):
            img = lif.get_image(i)

            info = {
                "index": i,
                "name": img.name,
                "x_size": getattr(img.dims, "x", None),
                "y_size": getattr(img.dims, "y", None),
                "z_slices": getattr(img.dims, "z", None),
                "t_frames": getattr(img.dims, "t", None),
                "channels": getattr(img, "channels", None),
                "scale": getattr(img, "scale", None),
            }
            file_results.append(info)

            if verbose:
                print(f"Index [{i}]: {info['name']}")
                print(f"  - Resolution: {info['x_size']} x {info['y_size']} pixels")
                print(f"  - Z-slices  : {info['z_slices']} planes")
                print(f"  - Time      : {info['t_frames']} frames")
                print(f"  - Channels  : {info['channels']} channels")
                print(f"  - Scale     : {info['scale']}")
                print("-" * 70)

        return file_results

    except Exception as e:
        print(f"[ERROR] Failed to read file: {lif_path}")
        print(f"Reason: {e}")
        return None


def inspect_multiple_lif(lif_file_list, base_dir=None, verbose=True):
    all_results = {}

    if base_dir is not None:
        base_dir = Path(base_dir)

    for file_item in lif_file_list:
        file_path = Path(file_item)

        if not file_path.is_absolute() and base_dir is not None:
            file_path = base_dir / file_path

        result = inspect_single_lif(file_path, verbose=verbose)
        all_results[str(file_path)] = result

    return all_results


## 3. 시각화 / 공통 보조 함수

표시용 정규화, 파일명 정리, dtype 변환, GIF 생성, scale metadata 변환 함수를 모읍니다.


In [ ]:
def normalize_for_view(data, p_low=2, p_high=99.8):
    data = np.asarray(data)

    if data.size == 0:
        return np.zeros_like(data, dtype=np.float32)

    if np.all(data == data.flat[0]):
        return np.zeros_like(data, dtype=np.float32)

    low, high = np.percentile(data, (p_low, p_high))

    if low == high:
        return np.zeros_like(data, dtype=np.float32)

    return exposure.rescale_intensity(
        data,
        in_range=(low, high),
        out_range=(0, 1)
    ).astype(np.float32)


def sanitize_name(name):
    bad_chars = ['/', '\\', ':', '*', '?', '"', '<', '>', '|']
    out = str(name)
    for ch in bad_chars:
        out = out.replace(ch, "_")
    return out.strip()


def ensure_uint16(arr):
    arr = np.asarray(arr)

    if np.issubdtype(arr.dtype, np.uint16):
        return arr

    if np.issubdtype(arr.dtype, np.integer):
        return np.clip(arr, 0, np.iinfo(np.uint16).max).astype(np.uint16)

    if np.issubdtype(arr.dtype, np.floating):
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        arr = np.clip(arr, 0, np.iinfo(np.uint16).max)
        return arr.astype(np.uint16)

    return arr.astype(np.uint16)


def ensure_uint8(arr):
    arr = np.asarray(arr)

    if np.issubdtype(arr.dtype, np.uint8):
        return arr

    if np.issubdtype(arr.dtype, np.integer):
        return np.clip(arr, 0, np.iinfo(np.uint8).max).astype(np.uint8)

    if np.issubdtype(arr.dtype, np.floating):
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        arr = np.clip(arr, 0, np.iinfo(np.uint8).max)
        return arr.astype(np.uint8)

    return arr.astype(np.uint8)


def stack_to_gif_bytes(stack, duration_ms=120, color=None):
    stack = np.asarray(stack)

    if stack.ndim != 3:
        raise ValueError(f"Expected stack shape (Z, Y, X), got {stack.shape}")

    frames = []
    for z_idx in range(stack.shape[0]):
        frame_norm = normalize_for_view(stack[z_idx], p_low=1, p_high=99.8)
        frame_uint8 = ensure_uint8(np.round(frame_norm * 255.0))

        if color is None:
            frames.append(Image.fromarray(frame_uint8, mode="L"))
            continue

        color = color.lower()
        frame_rgb = np.zeros((*frame_uint8.shape, 3), dtype=np.uint8)
        if color == "red":
            frame_rgb[..., 0] = frame_uint8
        elif color == "green":
            frame_rgb[..., 1] = frame_uint8
        elif color == "blue":
            frame_rgb[..., 2] = frame_uint8
        else:
            raise ValueError(f"Unsupported GIF color: {color}")
        frames.append(Image.fromarray(frame_rgb, mode="RGB"))

    if not frames:
        return b""

    buffer = BytesIO()
    frames[0].save(
        buffer,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,
    )
    return buffer.getvalue()


def make_reference_target_merged_gif_html(
    channel_data,
    channel_labels=("DAPI", "Reference", "Target"),
    duration_ms=120,
    title_prefix="",
    display_width=280
):
    ref_stack = np.asarray(channel_data[channel_labels[1]]["stack"])
    target_stack = np.asarray(channel_data[channel_labels[2]]["stack"])

    if ref_stack.shape != target_stack.shape:
        raise ValueError("Reference and Target stack shapes must match for GIF display")

    ref_frames = []
    target_frames = []
    merged_frames = []

    for z_idx in range(ref_stack.shape[0]):
        ref_norm = normalize_for_view(ref_stack[z_idx], p_low=1, p_high=99.8)
        target_norm = normalize_for_view(target_stack[z_idx], p_low=1, p_high=99.8)

        ref_uint8 = ensure_uint8(np.round(ref_norm * 255.0))
        target_uint8 = ensure_uint8(np.round(target_norm * 255.0))

        merged_rgb = np.zeros((*ref_uint8.shape, 3), dtype=np.uint8)
        merged_rgb[..., 0] = target_uint8
        merged_rgb[..., 1] = ref_uint8

        ref_frames.append(ref_uint8)
        target_frames.append(target_uint8)
        merged_frames.append(merged_rgb)

    ref_gif = base64.b64encode(
        stack_to_gif_bytes(np.stack(ref_frames, axis=0), duration_ms=duration_ms, color="green")
    ).decode("ascii")
    target_gif = base64.b64encode(
        stack_to_gif_bytes(np.stack(target_frames, axis=0), duration_ms=duration_ms, color="red")
    ).decode("ascii")

    merged_pil_frames = [Image.fromarray(frame, mode="RGB") for frame in merged_frames]
    merged_buffer = BytesIO()
    merged_pil_frames[0].save(
        merged_buffer,
        format="GIF",
        save_all=True,
        append_images=merged_pil_frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,
    )
    merged_gif = base64.b64encode(merged_buffer.getvalue()).decode("ascii")

    image_style = f"width:{display_width}px;height:auto;display:block;"
    merged_image_style = f"width:{display_width * 2}px;max-width:100%;height:auto;display:block;"
    title_html = f"<div style='color:white;font-weight:bold;margin-bottom:10px;'>{title_prefix} | Z-stack GIF</div>" if title_prefix else ""
    return f"""
<div style='background:black;padding:12px 12px 6px 12px;border-radius:8px;'>
  {title_html}
  <div style='display:grid;grid-template-columns:repeat(2, max-content);gap:18px;align-items:start;justify-content:start;'>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Reference</div><img src='data:image/gif;base64,{ref_gif}' style='{image_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Target</div><img src='data:image/gif;base64,{target_gif}' style='{image_style}' /></div>
    <div style='grid-column:1 / span 2;justify-self:start;'><div style='color:white;text-align:center;margin-bottom:6px;'>Merged</div><img src='data:image/gif;base64,{merged_gif}' style='{merged_image_style}' /></div>
  </div>
</div>
"""


def show_reference_target_z_gif_triplet(
    channel_data,
    title_prefix="",
    channel_labels=("DAPI", "Reference", "Target"),
    duration_ms=120,
    display_width=280
):
    html = make_reference_target_merged_gif_html(
        channel_data=channel_data,
        channel_labels=channel_labels,
        duration_ms=duration_ms,
        title_prefix=title_prefix,
        display_width=display_width,
    )
    display(HTML(html))


def safe_scale_to_dict(scale):
    if scale is None:
        return None

    try:
        if isinstance(scale, dict):
            return scale

        if isinstance(scale, (list, tuple)):
            return {
                "x": scale[0] if len(scale) > 0 else None,
                "y": scale[1] if len(scale) > 1 else None,
                "z": scale[2] if len(scale) > 2 else None,
                "t": scale[3] if len(scale) > 3 else None,
            }

        return {"raw": str(scale)}
    except Exception:
        return {"raw": str(scale)}


def get_scale_bar_length_pixels(img, bar_length_um=100):
    scale = getattr(img, "scale", None)
    if scale is None:
        return None

    try:
        pixel_size = scale[0]
        if pixel_size is None or pixel_size == 0:
            return None
        return bar_length_um / pixel_size
    except Exception:
        return None


## 4. stack / MIP 생성

LIF image 객체에서 채널별 Z-stack과 maximum intensity projection을 만드는 함수입니다.


In [ ]:
def get_full_stack(img, channel_idx):
    """
    특정 채널의 전체 3D stack 반환
    shape: (Z, Y, X)
    """
    z_slices = getattr(img.dims, "z", 1)
    stack = []

    for z in range(z_slices):
        frame = np.array(img.get_frame(z=z, t=0, c=channel_idx))
        stack.append(frame)

    return np.stack(stack, axis=0)


def get_max_projection(img, channel_idx):
    stack = get_full_stack(img, channel_idx)
    return np.max(stack, axis=0)


def make_channel_data(img, channel_order=(0, 1, 2), channel_labels=("DAPI", "Reference", "Target")):
    """
    각 채널에 대해 3D stack과 표시용 MIP를 생성
    """
    channel_data = {}

    for ch_idx, ch_label in zip(channel_order, channel_labels):
        stack = get_full_stack(img, ch_idx)
        mip = np.max(stack, axis=0)

        channel_data[ch_label] = {
            "stack": stack,
            "mip": mip,
        }

    return channel_data


## 5. metadata 구성

저장할 series-level metadata JSON 구조를 구성합니다.


In [ ]:
def build_series_metadata(
    lif_name,
    series_info,
    img,
    channel_order=(0, 1, 2),
    channel_labels=("DAPI", "Reference", "Target")
):
    return {
        "lif_name": lif_name,
        "series_name": series_info["name"],
        "series_index": series_info["index"],
        "x_size": series_info.get("x_size"),
        "y_size": series_info.get("y_size"),
        "z_slices": series_info.get("z_slices"),
        "t_frames": series_info.get("t_frames"),
        "channels": series_info.get("channels"),
        "scale": safe_scale_to_dict(series_info.get("scale")),
        "channel_order": list(channel_order),
        "channel_labels": list(channel_labels),
        "dims_from_image": {
            "x": getattr(img.dims, "x", None),
            "y": getattr(img.dims, "y", None),
            "z": getattr(img.dims, "z", None),
            "t": getattr(img.dims, "t", None),
        }
    }


## 6. TIFF / metadata 저장

채널별 TIFF stack과 metadata.json을 output 폴더에 저장합니다.


In [ ]:
def save_channel_tiffs_for_series(
    base_folder,
    lif_name,
    series_name,
    channel_data,
    metadata,
    channel_labels=("DAPI", "Reference", "Target"),
    save_stacks=True
):
    """
    저장 구조:
    base_folder/
      lif_name/
        series_name/
          stacks/
            DAPI_stack.tif
            Reference_stack.tif
            Target_stack.tif
          metadata.json
    """
    base_folder = Path(base_folder)
    lif_dir = base_folder / sanitize_name(lif_name)
    series_dir = lif_dir / sanitize_name(series_name)

    stacks_dir = series_dir / "stacks"

    series_dir.mkdir(parents=True, exist_ok=True)

    if save_stacks:
        stacks_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = {
        "stacks": {},
        "metadata": None,
    }

    for ch_label in channel_labels:
        if ch_label not in channel_data:
            continue

        stack = channel_data[ch_label]["stack"]

        if save_stacks:
            stack_path = stacks_dir / f"{sanitize_name(ch_label)}_stack.tif"
            io.imsave(str(stack_path), ensure_uint16(stack), check_contrast=False)
            saved_paths["stacks"][ch_label] = stack_path

    metadata_path = series_dir / "metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    saved_paths["metadata"] = metadata_path
    cleanup_dot_underscore_files(series_dir)
    return saved_paths


## 7. merge figure 표시

DAPI/Reference/Target/Merged MIP 패널과 scale bar를 notebook에 표시합니다.


In [ ]:
def show_single_series_figure(
    lif_name,
    series_name,
    img,
    channel_order=(0, 1, 2),
    channel_labels=("DAPI", "Reference", "Target"),
    figsize=(20, 5),
    bar_length_um=100,
    scale_bar_thickness=10,
    scale_bar_fontsize=16,
    precomputed_channel_data=None
):
    if precomputed_channel_data is None:
        channel_data = make_channel_data(
            img=img,
            channel_order=channel_order,
            channel_labels=channel_labels
        )
    else:
        channel_data = precomputed_channel_data

    ch_dapi = channel_data[channel_labels[0]]["mip"]
    ch_ref = channel_data[channel_labels[1]]["mip"]
    ch_target = channel_data[channel_labels[2]]["mip"]

    ch_dapi_norm = normalize_for_view(ch_dapi)
    ch_ref_norm = normalize_for_view(ch_ref)
    ch_target_norm = normalize_for_view(ch_target)

    merged = np.zeros((*ch_dapi.shape, 3), dtype=np.float32)
    merged[..., 0] = ch_target_norm
    merged[..., 1] = ch_ref_norm
    merged[..., 2] = ch_dapi_norm

    fig_title = f"{lif_name}_{series_name}"

    fig, axes = plt.subplots(1, 4, figsize=figsize, facecolor="black")
    fig.suptitle(fig_title, color="white", fontsize=20, fontweight="bold", y=0.95)

    imgs_for_plot = [ch_dapi_norm, ch_ref_norm, ch_target_norm, merged]
    cmaps_for_plot = [cmap_blue, cmap_green, cmap_red, None]
    titles_for_plot = [
        channel_labels[0],
        channel_labels[1],
        channel_labels[2],
        "Merged",
    ]

    scale_bar_len = get_scale_bar_length_pixels(img, bar_length_um=bar_length_um)

    for ax, img_plot, cmap, title in zip(axes, imgs_for_plot, cmaps_for_plot, titles_for_plot):
        ax.imshow(img_plot, cmap=cmap)
        ax.set_title(title, color="white", fontsize=17)
        ax.axis("off")

        if scale_bar_len is not None:
            scalebar = AnchoredSizeBar(
                ax.transData,
                scale_bar_len,
                f"{bar_length_um} µm",
                "lower right",
                pad=0.6,
                borderpad=0.8,
                sep=6,
                color="white",
                frameon=False,
                size_vertical=scale_bar_thickness,
                fontproperties={"size": scale_bar_fontsize}
            )
            ax.add_artist(scalebar)

    plt.tight_layout()
    plt.subplots_adjust(top=0.82)
    plt.show()

    return channel_data


## 8. 표시용 MIP 기반 histogram / FFT figure

저장된 파일이 아니라 메모리의 MIP로 histogram과 Fourier diagnostic figure를 표시합니다.


In [ ]:
def compute_fft_log_image(img):
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    return np.log1p(mag)


def show_histogram_and_fft_from_channel_data(
    channel_data,
    title_prefix="",
    channel_labels=("DAPI", "Reference", "Target"),
    figsize=(15, 15)
):
    """
    저장하지 않은 표시용 MIP 배열로 histogram / FFT figure를 표시한다.
    """
    channel_to_cmap = {
        channel_labels[0]: "Blues",
        channel_labels[1]: "Greens",
        channel_labels[2]: "Reds",
    }

    fig, axes = plt.subplots(3, 3, figsize=figsize, facecolor="black")
    fig.suptitle(f"{title_prefix} | Image / Histogram / Fourier", color="white", fontsize=18, y=0.92)

    for row_idx, ch_name in enumerate(channel_labels):
        img = channel_data[ch_name]["mip"]
        img_disp = normalize_for_view(img)
        fft_img = compute_fft_log_image(img)
        fft_disp = normalize_for_view(fft_img, p_low=0, p_high=99.9)

        # image
        ax = axes[row_idx, 0]
        ax.imshow(img_disp, cmap=channel_to_cmap[ch_name])
        ax.set_title(f"{ch_name} Image", color="white", fontsize=14)
        ax.axis("off")

        # histogram
        ax = axes[row_idx, 1]
        ax.hist(img.ravel(), bins=256, color="white")
        ax.set_title(f"{ch_name} Histogram", color="white", fontsize=14)
        ax.set_facecolor("black")
        ax.tick_params(colors="white")
        for spine in ax.spines.values():
            spine.set_color("white")
        ax.set_xlabel("Intensity", color="white")
        ax.set_ylabel("Count", color="white")

        # fourier
        ax = axes[row_idx, 2]
        ax.imshow(fft_disp, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{ch_name} Fourier Space", color="white", fontsize=14)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## 9. parsed results 기반 실행

metadata 필터링 결과를 바탕으로 series별 시각화, 저장, GIF 생성을 실행합니다.


In [ ]:
def visualize_and_save_matched_series(
    results,
    base_folder,
    name_filter="63x",
    channel_order=(0, 1, 2),
    channel_labels=("DAPI", "Reference", "Target"),
    figsize=(20, 5),
    bar_length_um=100,
    scale_bar_thickness=10,
    scale_bar_fontsize=16,
    show_merge_figure=True,
    save_tiffs=True,
    show_hist_fft=True,
    save_stacks=True,
    show_z_gif_triplet=True,
    gif_duration_ms=120
):
    matched_total = 0
    matched_details = {}

    for lif_path, series_list in results.items():
        if series_list is None:
            continue

        lif = LifFile(lif_path)
        lif_name = Path(lif_path).stem
        matched_details.setdefault(lif_name, [])

        for s in series_list:
            idx = s["index"]
            series_name = s["name"]

            if name_filter and name_filter.lower() not in series_name.lower():
                continue

            matched_total += 1
            img = lif.get_image(idx)

            # 3D stack + 표시용 MIP 생성
            channel_data = make_channel_data(
                img=img,
                channel_order=channel_order,
                channel_labels=channel_labels
            )

            # metadata 생성
            metadata = build_series_metadata(
                lif_name=lif_name,
                series_info=s,
                img=img,
                channel_order=channel_order,
                channel_labels=channel_labels
            )

            # merge figure
            if show_merge_figure:
                _ = show_single_series_figure(
                    lif_name=lif_name,
                    series_name=series_name,
                    img=img,
                    channel_order=channel_order,
                    channel_labels=channel_labels,
                    figsize=figsize,
                    bar_length_um=bar_length_um,
                    scale_bar_thickness=scale_bar_thickness,
                    scale_bar_fontsize=scale_bar_fontsize,
                    precomputed_channel_data=channel_data
                )

            saved_paths = None

            if save_tiffs:
                saved_paths = save_channel_tiffs_for_series(
                    base_folder=base_folder,
                    lif_name=lif_name,
                    series_name=series_name,
                    channel_data=channel_data,
                    metadata=metadata,
                    channel_labels=channel_labels,
                    save_stacks=save_stacks
                )

            if show_hist_fft:
                show_histogram_and_fft_from_channel_data(
                    channel_data=channel_data,
                    title_prefix=f"{lif_name}_{series_name}",
                    channel_labels=channel_labels
                )

            if show_z_gif_triplet:
                show_reference_target_z_gif_triplet(
                    channel_data=channel_data,
                    title_prefix=f"{lif_name}_{series_name}",
                    channel_labels=channel_labels,
                    duration_ms=gif_duration_ms
                )

            tiff_names = []
            if saved_paths is not None:
                tiff_names = [Path(path).name for path in saved_paths["stacks"].values()]

            matched_details[lif_name].append({
                "series_name": series_name,
                "series_index": idx,
                "tiff_names": tiff_names,
            })

    if matched_total == 0:
        print("[INFO] 조건에 맞는 series가 없습니다.")
        print(f"       조건: series_name contains '{name_filter}'")
    else:
        print(f"[INFO] 처리한 series 개수: {matched_total}")
        for lif_name, items in matched_details.items():
            if not items:
                continue
            print(f"[INFO] LIF 파일: {lif_name} | matched series: {len(items)}")
            for item in items:
                print(
                    f"       - series_index={item['series_index']}, "
                    f"series_name={item['series_name']}, "
                    f"tiffs={item['tiff_names']}"
                )


## 10. 전체 pipeline

여러 LIF 파일을 읽고 위 단계들을 한 번에 실행하는 wrapper입니다.


In [ ]:
def run_lif_pipeline(
    lif_files,
    base_dir=None,
    name_filter="63x",
    channel_order=(0, 1, 2),
    channel_labels=("DAPI", "Reference", "Target"),
    figsize=(20, 5),
    bar_length_um=100,
    scale_bar_thickness=10,
    scale_bar_fontsize=16,
    verbose=True,
    show_merge_figure=True,
    save_tiffs=True,
    show_hist_fft=True,
    save_stacks=True,
    show_z_gif_triplet=True,
    gif_duration_ms=120
):
    results = inspect_multiple_lif(
        lif_file_list=lif_files,
        base_dir=base_dir,
        verbose=verbose
    )

    visualize_and_save_matched_series(
        results=results,
        base_folder=base_dir,
        name_filter=name_filter,
        channel_order=channel_order,
        channel_labels=channel_labels,
        figsize=figsize,
        bar_length_um=bar_length_um,
        scale_bar_thickness=scale_bar_thickness,
        scale_bar_fontsize=scale_bar_fontsize,
        show_merge_figure=show_merge_figure,
        save_tiffs=save_tiffs,
        show_hist_fft=show_hist_fft,
        save_stacks=save_stacks,
        show_z_gif_triplet=show_z_gif_triplet,
        gif_duration_ms=gif_duration_ms
    )

    if base_dir is not None:
        cleanup_dot_underscore_files(base_dir)

    return results


## 11. Run Pipeline

현재 작업 디렉토리의 `data/` 폴더에서 `.lif` 파일을 자동으로 찾고 전체 pipeline을 실행합니다.


In [ ]:
# =========================================================
# 11. 동일 디렉토리의 data 폴더에서 LIF 파일 자동 실행
# =========================================================
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"data 폴더를 찾지 못했습니다: {DATA_DIR}")

lif_paths = sorted(DATA_DIR.glob("*.lif"))
if not lif_paths:
    raise FileNotFoundError(f"data 폴더 안에서 .lif 파일을 찾지 못했습니다: {DATA_DIR}")

lif_files = [p.name for p in lif_paths]
base_folder = DATA_DIR

print(f"DATA_DIR: {DATA_DIR}")
print(f"Found {len(lif_files)} LIF file(s):")
for name in lif_files:
    print(f"  - {name}")

results = run_lif_pipeline(
    lif_files=lif_files,
    base_dir=base_folder,
    name_filter="63x",
    channel_order=(0, 1, 2),   # (DAPI, Reference, Target)
    channel_labels=("DAPI", "Reference", "Target"),
    figsize=(20, 5),
    bar_length_um=1000,
    scale_bar_thickness=10,
    scale_bar_fontsize=16,
    verbose=True,
    show_merge_figure=True,
    save_tiffs=True,
    show_hist_fft=True,
    save_stacks=True,
    show_z_gif_triplet=True,
    gif_duration_ms=120,
)
